# PRC Dataset — Detailed Exploratory Analysis

**Task shape (inferred from schema):** predict `TAXITIME_SEC_mvt` (taxi time, in seconds) for departing flights.

- `training_YYYY-MM-01_YYYY-MM-01.parquet` — 12 monthly files (Jan 2025 → Jan 2026), fully labeled, 30 columns.
- `ranking.parquet` — same 30 columns; **ARR** rows are labeled, **DEP** rows have `TAXITIME_SEC_mvt` = NaN (these are the ones to predict). This is effectively the held-out/test set.
- `submitting.parquet` — submission template: `MVT_ID_mvt` + `TAXITIME_SEC_mvt` (all NaN), exactly matching the 215,876 DEP rows in `ranking.parquet`.

This notebook covers: data loading, schema/missingness, target distribution & data-quality issues, time-based patterns, categorical breakdowns, delay/schedule-adherence features, correlations, and a train-vs-test (covariate shift) comparison.

## 1. Setup

In [ ]:
import glob
import os

import matplotlib.pyplot as plt
import missingno as msno
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

DATA_DIR = "Dataset"
TRAIN_FILES = sorted(glob.glob(os.path.join(DATA_DIR, "training_*.parquet")))
TRAIN_FILES

## 2. Load data

In [ ]:
def month_tag(path):
    base = os.path.basename(path)
    return base.split("training_")[1].split("_")[0]


train_frames = []
for f in TRAIN_FILES:
    df = pd.read_parquet(f)
    df["source_month"] = month_tag(f)
    train_frames.append(df)

train = pd.concat(train_frames, ignore_index=True)
del train_frames

ranking = pd.read_parquet(os.path.join(DATA_DIR, "ranking.parquet"))
submitting = pd.read_parquet(os.path.join(DATA_DIR, "submitting.parquet"))

print("train:", train.shape)
print("ranking:", ranking.shape)
print("submitting:", submitting.shape)

In [ ]:
train.info(memory_usage="deep")

In [ ]:
train.head()

## 3. Sanity checks: ranking / submitting relationship

Confirm the inferred task structure holds: `submitting.parquet` IDs are exactly the DEP rows of `ranking.parquet` with a missing target.

In [ ]:
print("ranking PHASE_mvt counts:\n", ranking["PHASE_mvt"].value_counts())
print("\nranking TAXITIME_SEC_mvt nulls by phase:")
print(ranking.groupby("PHASE_mvt")["TAXITIME_SEC_mvt"].apply(lambda s: s.isna().mean()))

dep_ids = set(ranking.loc[ranking["PHASE_mvt"] == "DEP", "MVT_ID_mvt"])
sub_ids = set(submitting["MVT_ID_mvt"])
print("\nDEP rows in ranking:", len(dep_ids))
print("rows in submitting:", len(sub_ids))
print("exact match:", dep_ids == sub_ids)
print("submitting TAXITIME all-null:", submitting["TAXITIME_SEC_mvt"].isna().all())

## 4. Missing values

In [ ]:
missing = (
    train.drop(columns=["source_month"])
    .isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("pct_missing")
)
missing[missing["pct_missing"] > 0]

In [ ]:
msno.matrix(train.drop(columns=["source_month"]).sample(min(20000, len(train)), random_state=0))
plt.title("Missingness pattern (20k row sample)")
plt.show()

## 5. Target variable: `TAXITIME_SEC_mvt`

Only DEP rows are the prediction target in the ranking set, but training has both phases labeled — check whether taxi time behaves differently for DEP (taxi-out) vs ARR (taxi-in), and look for data-quality issues (e.g. negative values seen in `describe()`).

In [ ]:
train["TAXITIME_SEC_mvt"].describe()

In [ ]:
neg = train[train["TAXITIME_SEC_mvt"] < 0]
extreme = train[train["TAXITIME_SEC_mvt"] > train["TAXITIME_SEC_mvt"].quantile(0.999)]
print(f"negative taxitime rows: {len(neg)} ({len(neg)/len(train):.4%})")
print(f"top-0.1%% extreme rows: {len(extreme)}, min={extreme['TAXITIME_SEC_mvt'].min()}, max={extreme['TAXITIME_SEC_mvt'].max()}")
neg[["PHASE_mvt", "ADEP_mvt", "ADES_mvt", "AIRCRAFT_TYPE_mvt", "TAXITIME_SEC_mvt"]].head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

clean = train[(train["TAXITIME_SEC_mvt"] >= 0) & (train["TAXITIME_SEC_mvt"] < 3000)]
sns.histplot(data=clean, x="TAXITIME_SEC_mvt", hue="PHASE_mvt", bins=80, kde=True, ax=axes[0])
axes[0].set_title("Taxi time distribution (0-3000s, outliers/negatives excluded)")

sns.boxplot(data=clean, x="PHASE_mvt", y="TAXITIME_SEC_mvt", ax=axes[1])
axes[1].set_title("Taxi time by phase")
plt.tight_layout()
plt.show()

## 6. Time-based patterns

In [ ]:
train["sched_hour"] = train["SCHED_TIME_UTC_mvt"].dt.hour
train["sched_dow"] = train["SCHED_TIME_UTC_mvt"].dt.day_name()
train["sched_month"] = train["SCHED_TIME_UTC_mvt"].dt.to_period("M").astype(str)

dep = train[(train["PHASE_mvt"] == "DEP") & (train["TAXITIME_SEC_mvt"].between(0, 3000))]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

dep.groupby("sched_month")["TAXITIME_SEC_mvt"].median().plot(kind="bar", ax=axes[0])
axes[0].set_title("Median DEP taxi time by month")
axes[0].tick_params(axis="x", rotation=45)

dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dep.groupby("sched_dow")["TAXITIME_SEC_mvt"].median().reindex(dow_order).plot(kind="bar", ax=axes[1])
axes[1].set_title("Median DEP taxi time by day of week")

dep.groupby("sched_hour")["TAXITIME_SEC_mvt"].median().plot(kind="line", marker="o", ax=axes[2])
axes[2].set_title("Median DEP taxi time by hour of day (UTC)")

plt.tight_layout()
plt.show()

## 7. Categorical breakdowns

In [ ]:
def top_n_median_plot(df, col, target="TAXITIME_SEC_mvt", n=15, min_count=200):
    counts = df[col].value_counts()
    top = counts[counts >= min_count].head(n).index
    sub = df[df[col].isin(top)]
    order = sub.groupby(col)[target].median().sort_values(ascending=False).index
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.boxplot(data=sub, x=col, y=target, order=order, ax=ax, showfliers=False)
    ax.set_title(f"{target} by {col} (top {n} by frequency, min {min_count} obs)")
    ax.tick_params(axis="x", rotation=60)
    plt.tight_layout()
    plt.show()
    print(counts.head(n))


top_n_median_plot(dep, "ADEP_mvt")

In [ ]:
top_n_median_plot(dep, "AIRCRAFT_TYPE_mvt")

In [ ]:
top_n_median_plot(dep, "RUNWAY_mvt")

In [ ]:
top_n_median_plot(dep, "MARKET_SEGMENT_flt", min_count=50)

In [ ]:
top_n_median_plot(dep, "AIRCRAFT_OPERATOR_flt")

In [ ]:
for col in ["FLIGHT_RULE_mvt", "FLIGHT_TYPE_flt", "WK_TBL_CAT_flt"]:
    print(f"\n=== {col} ===")
    print(dep.groupby(col)["TAXITIME_SEC_mvt"].agg(["count", "median", "mean", "std"]))

## 8. Schedule adherence / delay features

Derive delay measures from the timestamp columns and see how they relate to taxi time — e.g. does a late pushback (`BLOCK_TIME` vs `SCHED_TIME`) correlate with longer or shorter taxi-out times (congestion vs off-peak effects)?

> ⚠️ **Correction — see §12. `pushback_delay_min` below cannot be used as a model feature.**
>
> It is derived from `BLOCK_TIME_UTC_mvt`, which is **100% blank in `ranking.parquet`**. The target is
> *exactly* `MVT_TIME_UTC_mvt - BLOCK_TIME_UTC_mvt` (100.000% match on train DEP rows), so the organizers
> removed the off-block time — that is the whole prediction task.
>
> The cells below remain valid as a *description* of the training data, but any delay feature has to be
> rebuilt from `AOBT_3_flt` / `EOBT_1_flt`, which do survive into the ranking set. See
> `src.cleaning.add_offblock_proxies`.

In [ ]:
dep = dep.copy()
dep["pushback_delay_min"] = (dep["BLOCK_TIME_UTC_mvt"] - dep["SCHED_TIME_UTC_mvt"]).dt.total_seconds() / 60
dep["eobt_delay_min"] = (dep["IOBT_flt"] - dep["EOBT_1_flt"]).dt.total_seconds() / 60

dep[["pushback_delay_min", "eobt_delay_min", "TAXITIME_SEC_mvt"]].describe()

In [ ]:
sample = dep.dropna(subset=["pushback_delay_min"])
sample = sample[sample["pushback_delay_min"].between(-60, 180)]
sample = sample.sample(min(30000, len(sample)), random_state=0)

sns.scatterplot(data=sample, x="pushback_delay_min", y="TAXITIME_SEC_mvt", alpha=0.15, s=10)
plt.title("Pushback delay vs taxi-out time")
plt.show()

print("correlation:", sample["pushback_delay_min"].corr(sample["TAXITIME_SEC_mvt"]))

## 9. Correlation overview (numeric / engineered features)

In [ ]:
num_cols = ["TAXITIME_SEC_mvt", "pushback_delay_min", "eobt_delay_min", "sched_hour"]
corr = dep[num_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix")
plt.show()

## 10. Train vs. ranking (test) — covariate shift check

Compare feature distributions between the labeled training data and the DEP rows of `ranking.parquet` we need to predict, to flag any distribution shift that would undermine a model trained purely on the monthly files.

In [ ]:
rank_dep = ranking[ranking["PHASE_mvt"] == "DEP"].copy()
rank_dep["sched_hour"] = rank_dep["SCHED_TIME_UTC_mvt"].dt.hour
rank_dep["sched_month"] = rank_dep["SCHED_TIME_UTC_mvt"].dt.to_period("M").astype(str)

print("ranking DEP date range:", rank_dep["SCHED_TIME_UTC_mvt"].min(), "→", rank_dep["SCHED_TIME_UTC_mvt"].max())
print("training date range:", train["SCHED_TIME_UTC_mvt"].min(), "→", train["SCHED_TIME_UTC_mvt"].max())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dep["sched_hour"].value_counts(normalize=True).sort_index().plot(kind="bar", ax=axes[0], alpha=0.6, label="train")
rank_dep["sched_hour"].value_counts(normalize=True).sort_index().plot(kind="bar", ax=axes[0], alpha=0.6, label="ranking (test)", color="orange")
axes[0].set_title("Scheduled hour distribution")
axes[0].legend()

top_airports = dep["ADEP_mvt"].value_counts().head(10).index
train_pct = dep[dep["ADEP_mvt"].isin(top_airports)]["ADEP_mvt"].value_counts(normalize=True)
rank_pct = rank_dep[rank_dep["ADEP_mvt"].isin(top_airports)]["ADEP_mvt"].value_counts(normalize=True)
cmp_df = pd.DataFrame({"train": train_pct, "ranking": rank_pct}).fillna(0)
cmp_df.plot(kind="bar", ax=axes[1])
axes[1].set_title("Top-10 ADEP airport share: train vs ranking")

plt.tight_layout()
plt.show()

## 11. Summary of findings

_Fill in after running the notebook — key points to capture:_
- Overall taxi-time distribution shape, and the size/impact of the negative-value data-quality issue.
- Whether DEP vs ARR taxi times differ meaningfully (only DEP is predicted).
- Strongest categorical drivers (airport, runway, aircraft type, operator).
- Any temporal seasonality (month/day-of-week/hour) worth encoding as features.
- Whether pushback/EOBT delay is predictive of taxi time.
- Whether train and ranking (test) distributions line up, or whether reweighting/careful validation-split design is needed.

---

## 12. Pipeline modules: cleaning + external data

Everything below runs through the reusable modules in `src/`, so the identical logic applies to the
training files and to `ranking.parquet`. Four findings from the reconnaissance pass drive the design:

**1. The task is "infer the off-block time, given the takeoff time."**
`TAXITIME_SEC_mvt == MVT_TIME_UTC_mvt - BLOCK_TIME_UTC_mvt` exactly. `BLOCK_TIME_UTC_mvt` is 100% blank in
the ranking set, while `MVT_TIME_UTC_mvt` (actual takeoff) is 100% present. So the **takeoff time is a
legitimate, fully-available feature** — congestion/queue features and weather joins should all be anchored
on it, and anything built from the off-block time is unusable at prediction time.

**2. `AOBT_3_flt` is the dominant field.** The ADS-B-derived off-block time is present in 98.5% of ranking
DEP rows; used raw as a prediction it already scores RMSE ≈ 385 s against a target with σ = 546 s. Its bias
is strongly airport-specific (LTFM +236 s, LIRF −214 s), which a model can correct.

**3. A ~0.03% tail dominates an RMSE metric.** Against a per-airport-median baseline the 584 DEP rows over
2 h carry **46% of all squared error**. They are date/recording errors, not long taxis.

**4. That tail has a crisp signature — a missing `AIRCRAFT_TYPE_mvt`.** 75% of rows over 2 h lack an
aircraft type vs 0.07% of rows overall (~1000× enrichment, almost all at LIRF). These are a separate regime
(n = 1,507, mean 6,451 s / σ 11,170 s vs 987 s / 432 s), and **`ranking.parquet` contains 108 of them**, so
they must still be predicted rather than ignored.

**External data** (license-vetted in `SKILLS.md` §1): hourly weather from **Open-Meteo** (CC BY 4.0) and
public holidays from the **`holidays`** package (MIT). Cache them once with
`python scripts/fetch_external.py`.

In [ ]:
import sys

sys.path.insert(0, ".")

import numpy as np
import pandas as pd

from src.airports import AIRPORTS, DEPARTURE_AIRPORTS
from src.cleaning import (
    MAX_PLAUSIBLE_TAXI_SEC,
    add_offblock_proxies,
    clean_movements,
    cleaning_report,
    load_ranking,
    load_training,
    trainable_mask,
)
from src.external_data import (
    WEATHER_ATTRIBUTION,
    attach_holidays,
    attach_weather,
    load_holidays,
    load_weather,
)

weather = load_weather()          # cached by scripts/fetch_external.py
holidays_cal = load_holidays()

print("weather :", weather.shape, weather["time_utc"].min(), "->", weather["time_utc"].max())
print("holidays:", holidays_cal.shape)
print(WEATHER_ATTRIBUTION)
AIRPORTS

### 12.1 Cleaning

`clean_movements` is deliberately non-destructive: it repairs only what is provably an artefact (the 369
negative taxi times, all between −12 s and −1 s, are clock rounding and get clipped to 0; the literal `"NA"`
runway token becomes a real null), and *flags* everything merely suspicious. Row exclusion is a separate
decision made by `trainable_mask`, because the same cleaning function must run over `ranking.parquet`
where nothing may be dropped.

In [ ]:
train_raw = load_training()
dep = add_offblock_proxies(clean_movements(train_raw, phase="DEP"))

cleaning_report(train_raw, dep).style.format({"share": "{:.4%}"})

In [ ]:
# Excluding the untrustworthy labels is what actually matters for an RMSE metric:
tm = trainable_mask(dep)
print(f"trainable: {tm.sum():,} / {len(dep):,}  (excluded {(~tm).sum():,} = {(~tm).mean():.4%})")
print()
print(pd.DataFrame({
    "all DEP rows": dep["TAXITIME_SEC_mvt"].describe(percentiles=[0.01, 0.5, 0.99, 0.999]),
    "trainable only": dep.loc[tm, "TAXITIME_SEC_mvt"].describe(percentiles=[0.01, 0.5, 0.99, 0.999]),
}).to_string(float_format=lambda x: f"{x:,.1f}"))

In [ ]:
# Why it matters: share of total squared error carried by the tail, against a per-airport-median baseline.
y = dep["TAXITIME_SEC_mvt"]
sq = (y - dep.groupby("ADEP_mvt")["TAXITIME_SEC_mvt"].transform("median")) ** 2

rows = []
for thr in [3600, 7200, 10800, 20000]:
    m = y > thr
    rows.append({
        "target >": f"{thr:,}s",
        "n rows": int(m.sum()),
        "% of rows": f"{m.mean():.4%}",
        "% of squared error": f"{sq[m].sum() / sq.sum():.2%}",
        "baseline RMSE excl.": f"{np.sqrt(sq[~m].mean()):,.1f}s",
    })
print(f"baseline RMSE over ALL rows: {np.sqrt(sq.mean()):,.1f}s")
pd.DataFrame(rows)

In [ ]:
# Cleaning must be a no-op on row count for the ranking set.
rank_raw = load_ranking()
rank_dep = add_offblock_proxies(clean_movements(rank_raw, phase="DEP"))

n_expected = int((rank_raw["PHASE_mvt"] == "DEP").sum())
assert len(rank_dep) == n_expected, "ranking rows were dropped — submission would be rejected"
print(f"ranking DEP rows preserved: {len(rank_dep):,}")
print()
print("the anomalous regime is present in the ranking set too, and must be predicted:")
print(rank_dep.groupby("ADEP_mvt")["dq_no_aircraft_type"].sum().loc[lambda s: s > 0].to_string())

### 12.2 External data joins

Both externals join on the **takeoff time** (`MVT_TIME_UTC_mvt`) — the only timestamp available in both the
training files and the ranking set. Holidays additionally convert UTC to each airport's *local* civil date,
with the correct country **and subdivision**: Epiphany (6 Jan) is a public holiday at Munich (Bavaria),
Barcelona, Madrid and Rome, but **not** at Frankfurt (Hesse) — a national-level calendar would get that
wrong for two airports in the same country.

In [ ]:
dep = attach_holidays(attach_weather(dep, weather), holidays_cal)
rank_dep = attach_holidays(attach_weather(rank_dep, weather), holidays_cal)

for name, d in [("train DEP", dep), ("rank DEP", rank_dep)]:
    print(f"{name}: {len(d):,} rows | weather NaN {d['wx_visibility'].isna().mean():.4%} "
          f"| holiday rows {d['is_holiday'].mean():.2%}")

print()
print("holidays actually falling inside the ranking window (Jan + Jul 2026):")
print(rank_dep[rank_dep["is_holiday"]].groupby(["ADEP_mvt", "holiday_name"]).size().to_string())

In [ ]:
# Does the weather actually carry signal? The RAW marginal is confounded — low visibility happens on calm
# winter nights when traffic is light — so compare each flight to the median of its own
# (airport x hour-of-day x month) cell instead.
tj = dep[trainable_mask(dep)].copy()
tj["hour"] = tj["MVT_TIME_UTC_mvt"].dt.hour
tj["month"] = tj["MVT_TIME_UTC_mvt"].dt.month
tj["rel"] = tj["TAXITIME_SEC_mvt"] / tj.groupby(["ADEP_mvt", "hour", "month"])["TAXITIME_SEC_mvt"].transform("median")

vis_bins = pd.cut(tj["wx_visibility"], [0, 200, 500, 1000, 1500, 3000, 5000, 10000, 25000, 100000])
vis = tj.groupby(vis_bins, observed=True).agg(n=("rel", "size"), raw_median=("TAXITIME_SEC_mvt", "median"),
                                              rel_to_cell=("rel", "median"))
print("VISIBILITY (m) — raw median is non-monotonic, controlled ratio is clean:")
print(vis.to_string(float_format=lambda x: f"{x:.3f}"))

In [ ]:
winter = tj[tj["month"].isin([11, 12, 1, 2, 3])]
snow_bins = pd.cut(winter["wx_snowfall"], [-0.001, 0.0001, 0.1, 0.5, 1.0, 100])
print("SNOWFALL (cm/h), winter months, relative to the airport x hour x month median:")
print(winter.groupby(snow_bins, observed=True).agg(n=("rel", "size"), rel_to_cell=("rel", "median")).to_string(
    float_format=lambda x: f"{x:.3f}"))

print()
tj["deice_regime"] = (tj["wx_temperature_2m"] < 3) & (tj["wx_precipitation"] > 0)
print("DE-ICING REGIME (T < 3 C and precipitation > 0):")
print(tj.groupby("deice_regime").agg(n=("rel", "size"), raw_median=("TAXITIME_SEC_mvt", "median"),
                                     rel_to_cell=("rel", "median")).to_string(float_format=lambda x: f"{x:.3f}"))

print()
gust_bins = pd.cut(tj["wx_wind_gusts_10m"], [-0.1, 5, 10, 15, 20, 25, 100])
print("WIND GUSTS (m/s):")
print(tj.groupby(gust_bins, observed=True).agg(n=("rel", "size"), rel_to_cell=("rel", "median")).to_string(
    float_format=lambda x: f"{x:.3f}"))

### 12.3 What this establishes

Controlling for airport × hour × month, the weather signal is monotonic and physical:

| regime | effect on taxi time |
|---|---|
| visibility < 200 m | **1.07×** (and monotonic up the visibility scale) |
| snowfall 0–0.1 cm/h | **1.62×** |
| snowfall 0.1–0.5 cm/h | **1.74×** |
| snowfall 0.5–1.0 cm/h | **2.32×** |
| snowfall > 1.0 cm/h | **4.67×** |
| de-icing regime (T < 3 °C, precip > 0) | **1.16×** (11,803 rows) |
| wind gusts 15–20 m/s | 1.04× |

Note the raw visibility marginal *looks* flat-to-inverted — low visibility coincides with light-traffic
winter nights. Only the controlled comparison shows the real effect, which is a caution for the feature
selection step: judge candidate weather features on the controlled comparison, not the raw correlation.

Holidays are, honestly, a **weak** feature at this level (median 906 s on holidays vs 912 s otherwise) —
kept because they cost nothing and may interact with airport and season, not because the marginal impresses.

**Two scoring facts worth carrying into modelling:**

1. **January 2026 is 70% of the score** (152,719 of 215,876 rows), and **July 2026 contains only EDDF,
   EGLL and EHAM** — the other seven airports appear in January alone. A validation split should mirror
   that composition rather than sampling uniformly across the year.
2. The 108 missing-aircraft-type rows in the ranking set are irreducibly noisy, but predicting their
   *conditional mean* (~6,450 s) rather than a normal taxi time is worth roughly 15,000 MSE — on the order
   of 15–20 s of final RMSE. That is a larger swing than most feature engineering will produce, so it
   deserves an explicit branch in the model rather than being left to the regressor.

---

## 13. Feature engineering — the built feature set

Sections 1–11 described the raw data and §12 established cleaning and the external joins. This section is the
**feature layer**: what `src/features.py` produces, which columns are legitimate model inputs, and — for each
family — whether it actually carries signal.

Everything here reads the cached tables written by `python scripts/build_features.py`, so the section runs in
seconds rather than rebuilding 2 M rows.

**The rule that governs the whole layer** (see §12 and `docs/findings.md` finding 1): a feature is usable only if it is
populated in `ranking.parquet`. `BLOCK_TIME_UTC_mvt` is 100 % blank there and the target is exactly
`MVT_TIME_UTC_mvt − BLOCK_TIME_UTC_mvt`, so the off-block time *is* the prediction. The actual takeoff time is
fully available and is therefore a legitimate anchor — which is what makes the congestion and queue families
below computable at prediction time at all.

In [ ]:
import sys
sys.path.insert(0, ".")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from src.cleaning import trainable_mask
from src.features import CATEGORICAL_FEATURES, EXCLUDE_FROM_MODEL, model_features

TRAIN_F = "data/processed/train_features.parquet"
RANK_F = "data/processed/ranking_features.parquet"


def null_rate(path):
    "Null rates straight from the parquet row-group statistics - no need to load the data."
    md = pq.ParquetFile(path).metadata
    counts, n = {}, md.num_rows
    for rg in range(md.num_row_groups):
        for i in range(md.num_columns):
            col = md.row_group(rg).column(i)
            if col.statistics is not None:
                counts[col.path_in_schema] = counts.get(col.path_in_schema, 0) + col.statistics.null_count
    return pd.Series(counts) / n


schema = pq.ParquetFile(TRAIN_F).schema_arrow
train_nulls, rank_nulls = null_rate(TRAIN_F), null_rate(RANK_F)

print(f"train features : {pq.ParquetFile(TRAIN_F).metadata.num_rows:>9,} rows x {len(schema.names)} cols")
print(f"ranking features: {pq.ParquetFile(RANK_F).metadata.num_rows:>9,} rows x {len(pq.ParquetFile(RANK_F).schema_arrow.names)} cols")

### 13.1 The full inventory

Every column in the built table, grouped by the family that produced it. `avail_train` / `avail_rank` are the
**non-null** rates — the second one is what decides whether a feature is usable at all.

In [ ]:
def family(col):
    if col in ("TAXITIME_SEC_mvt",):                        return "0. target"
    if col.startswith("dq_"):                               return "6. data-quality flags"
    if col.startswith("proxy_taxi_") or col in ("offblock_delay_sec", "takeoff_vs_sched_sec"):
                                                            return "1. off-block proxies"
    if col.startswith("cong_"):                             return "2. congestion (window counts)"
    if col.startswith("queue_") or col.startswith("sec_"):  return "3. queue / sequence"
    if col.startswith("wind_") or col.startswith("gust_"):  return "8. wind components"
    if col.startswith("wx_"):                               return "7. weather"
    if col in ("local_minute_of_day", "local_month", "local_doy", "is_weekend", "local_hour",
               "local_dow", "local_time", "local_date") or col.endswith(("_sin", "_cos")):
                                                            return "4. time encodings"
    if col.startswith("is_") and "holiday" in col:          return "5. holidays"
    if col in ("holiday_name",):                            return "5. holidays"
    if col.startswith("stand_"):                            return "9. stand"
    if col in ("planned_enroute_sec", "is_longhaul", "dest_region", "dest_country", "is_domestic_region"):
                                                            return "10. route"
    return "11. raw source columns"


inv = pd.DataFrame({"column": schema.names})
inv["dtype"] = [str(schema.field(c).type) for c in inv["column"]]
inv["family"] = inv["column"].map(family)
inv["avail_train"] = 1 - inv["column"].map(train_nulls).fillna(0)
inv["avail_rank"] = 1 - inv["column"].map(rank_nulls).fillna(0)

nums, cats = model_features(pd.read_parquet(RANK_F))
model_cols = set(nums) | set(cats)
inv["model_input"] = inv["column"].isin(model_cols)

summary = (inv.groupby("family")
             .agg(n_cols=("column", "size"), n_model_inputs=("model_input", "sum"),
                  min_avail_rank=("avail_rank", "min"))
             .sort_index())
print(f"{len(model_cols)} of {len(inv)} columns are model inputs "
      f"({len(nums)} numeric + {len(cats)} categorical)\n")
summary.style.format({"min_avail_rank": "{:.3f}"})

In [ ]:
# The full listing, family by family. `avail_rank` below 1.0 is not disqualifying on its own -
# gradient boosting handles missing values - but a low value means the model will lean on the
# fallback for those rows, so it is worth knowing which ones they are.
pd.set_option("display.max_rows", 200)
(inv.sort_values(["family", "column"])
    .loc[:, ["family", "column", "dtype", "avail_train", "avail_rank", "model_input"]]
    .style.format({"avail_train": "{:.4f}", "avail_rank": "{:.4f}"})
    .hide(axis="index"))

### 13.2 What we can use, and what is deliberately withheld

27 of the 115 columns are **not** model inputs. Three different reasons, and they are worth separating because
only the first is about leakage in the usual sense.

In [ ]:
reasons = {
    "BLOCK_TIME_UTC_mvt": "target's other half - 100% blank in ranking.parquet",
    "TAXITIME_SEC_mvt": "the target itself",
    "ARVT_3_flt": "FLOWN arrival time - happens after the event",
    "dq_implausible_target": "computed FROM the target (> 7200s) - would leak the answer",
    "dq_was_negative": "computed FROM the target (< 0) - would leak the answer",
    "MVT_ID_mvt": "identifier", "FLIGHT_ID_mvt": "identifier",
    "FLIGHT_mvt": "identifier", "CALLSIGN_flt": "identifier",
    "PHASE_mvt": "constant (DEP) after filtering", "source_month": "bookkeeping",
    "local_time": "raw timestamp - encoded as time features instead",
    "local_date": "raw timestamp - encoded as time features instead",
    "holiday_name": "high-cardinality string - is_holiday carries it",
    "STAND_mvt": "superseded by stand_full / stand_area",
    "ADEP_flt": "captured by the dq_* mismatch flags",
    "ADES_FILED_flt": "captured by the dq_* mismatch flags",
    "wx_grid_lat": "weather grid bookkeeping", "wx_grid_lon": "weather grid bookkeeping",
    "wx_grid_elevation_m": "weather grid bookkeeping",
}
excl = inv.loc[~inv["model_input"], ["column", "dtype"]].copy()
excl["why"] = excl["column"].map(reasons).fillna("raw datetime - encoded as a duration instead")
excl["category"] = np.where(excl["column"].isin(
        ["BLOCK_TIME_UTC_mvt", "TAXITIME_SEC_mvt", "ARVT_3_flt",
         "dq_implausible_target", "dq_was_negative"]), "LEAKAGE", "not informative / superseded")
print(excl["category"].value_counts().to_string(), "\n")
excl.sort_values(["category", "column"]).style.hide(axis="index")

Two of those leakage exclusions were **caught late, in this feature set, not inherited from §12**:
`dq_implausible_target` and `dq_was_negative` are data-quality flags computed *from* `TAXITIME_SEC_mvt`. They
are useful for auditing and for `trainable_mask`, but as model inputs they hand the answer to the regressor and
cannot be computed at prediction time at all. `dq_no_aircraft_type` is the one flag that legitimately stays —
it reads `AIRCRAFT_TYPE_mvt`, which *is* populated in the ranking set.

In [ ]:
by_fam = (inv[inv["model_input"]].groupby("family")["column"]
            .apply(lambda s: ", ".join(sorted(s))).to_frame("features"))
for fam, row in by_fam.iterrows():
    n = len(row["features"].split(", "))
    print(f"\n{fam}  ({n})\n" + "-" * 78)
    print("  " + row["features"].replace(", ", ", "))

### 13.3 Does each family actually carry signal?

Judged the way §3's method note requires: **the controlled comparison**, not a raw correlation. Each flight is
divided by the median of its own (airport × hour-of-day × month) cell, so a ratio of **1.00 means the feature
adds nothing beyond airport, time of day and season** — which is precisely the confounding that made the raw
visibility marginal come out backwards in §12.2.

In [ ]:
cols = ["ADEP_mvt", "MVT_TIME_UTC_mvt", "TAXITIME_SEC_mvt", "AIRCRAFT_TYPE_mvt",
        "queue_at_pushback", "cong_dep_60", "cong_rwy_30", "sec_since_prev_dep_rwy",
        "wind_crosswind_ms", "planned_enroute_sec", "stand_area"]
t = pd.read_parquet(TRAIN_F, columns=cols)
t = t[trainable_mask(t)].copy()
t["hour"] = t["MVT_TIME_UTC_mvt"].dt.hour
t["month"] = t["MVT_TIME_UTC_mvt"].dt.month
t["rel"] = t["TAXITIME_SEC_mvt"] / t.groupby(["ADEP_mvt", "hour", "month"])["TAXITIME_SEC_mvt"].transform("median")

TESTS = [
    ("planned_enroute_sec",    [0, 3600, 7200, 3 * 3600, 6 * 3600, 10 * 3600, 1e6]),
    ("queue_at_pushback",      [-1, 2, 5, 8, 12, 16, 20, 100]),
    ("wind_crosswind_ms",      [-0.1, 2, 4, 6, 8, 12, 100]),
    ("sec_since_prev_dep_rwy", [0, 60, 90, 120, 180, 300, 600, 1e9]),
    ("cong_rwy_30",            [-1, 5, 10, 15, 20, 30, 100]),
    ("cong_dep_60",            [-1, 20, 40, 60, 80, 100, 200]),
]
for col, bins in TESTS:
    g = (t.groupby(pd.cut(t[col], bins), observed=True)
           .agg(n=("rel", "size"), raw_median=("TAXITIME_SEC_mvt", "median"), rel_to_cell=("rel", "median")))
    spread = g["rel_to_cell"].max() - g["rel_to_cell"].min()
    print(f"--- {col}   (spread {spread:.3f}) ---")
    print(g.to_string(float_format=lambda x: f"{x:,.3f}"), "\n")

**The result overturned the plan.** `SKILLS.md` had called takeoff-anchored congestion "the highest-value
feature family still to build". It is not:

| feature | controlled ratio, low → high | verdict |
|---|---|---|
| `planned_enroute_sec` | 0.874 → 1.131 | **strongest**, monotonic |
| `queue_at_pushback` | 0.963 → 1.095 | **strong**, monotonic |
| `wind_crosswind_ms` | 0.999 → 1.062 | modest, monotonic |
| `sec_since_prev_dep_rwy` | 0.946 → 1.021 | modest, **non-monotonic** |
| `cong_rwy_30` | 0.991 → 1.016 | weak |
| `cong_dep_60` | 0.994 → 1.008 | **flat** |

Counting movements in a ±N-minute window is almost entirely absorbed by the airport × hour control, because a
count of nearby traffic largely *is* a restatement of time of day. What works is measuring the **surface
state**: `queue_at_pushback` counts the flights that have already pushed back (`AOBT_3_flt`) but not yet taken
off (`MVT_TIME_UTC_mvt`) at the instant this aircraft pushes back — the queue it is physically joining. Both
timestamps survive into the ranking set, so it is computable at prediction time.

In [ ]:
# Independent sanity check on the queue feature: it should reproduce the per-airport taxi ordering
# without ever being told about it.
q = pd.read_parquet(TRAIN_F, columns=["ADEP_mvt", "queue_at_pushback", "TAXITIME_SEC_mvt",
                                      "AIRCRAFT_TYPE_mvt", "stand_area"])
q = q[trainable_mask(q)]
chk = (q.groupby("ADEP_mvt")
         .agg(median_queue=("queue_at_pushback", "median"),
              median_taxi=("TAXITIME_SEC_mvt", "median"))
         .sort_values("median_taxi", ascending=False))
print(chk.to_string(float_format=lambda x: f"{x:,.0f}"))
print(f"\nSpearman rank correlation queue vs taxi across airports: "
      f"{chk['median_queue'].corr(chk['median_taxi'], method='spearman'):.3f}")

The queue feature gets the **extremes** right — EGLL highest on both, LSZH lowest on both — but the rank
correlation across all ten airports is only **0.396**, and the exceptions are the informative part. LIRF has a
*short* queue (6) but the second-longest taxi (1,025 s); EHAM has a middling queue (11) and the second-shortest
taxi (742 s).

That is the correct result rather than a disappointing one, and it says something the aggregate cannot: taxi
time is roughly *distance ÷ speed + queueing*, and airports differ in *geometry* as much as in congestion.
Fiumicino's long taxiways produce long taxi times at low traffic; Schiphol's queue is real but its stands sit
close to the runways. A congestion feature alone can never separate those two regimes — which is exactly why
the stand and runway features end up carrying ~25 % of model importance in §13.6.

### 13.4 Will these transfer? Train vs ranking

A feature that is strong in 2025 is useless if it is distributed differently in 2026 or if its categories are
unseen. Both were checked.

In [ ]:
skew_cols = ["cong_dep_15", "cong_dep_60", "cong_arr_60", "cong_rwy_30", "queue_at_pushback",
             "queue_at_takeoff", "sec_since_prev_dep_rwy", "planned_enroute_sec",
             "wind_crosswind_ms", "proxy_taxi_aobt3", "proxy_taxi_eobt1"]
a = pd.read_parquet(TRAIN_F, columns=skew_cols)
b = pd.read_parquet(RANK_F, columns=skew_cols)
skew = pd.DataFrame({
    "train_median": a.median(), "rank_median": b.median(),
    "train_p90": a.quantile(0.9), "rank_p90": b.quantile(0.9),
    "avail_train": a.notna().mean(), "avail_rank": b.notna().mean(),
})
skew["median_ratio"] = skew["rank_median"] / skew["train_median"]
skew.style.format({"train_median": "{:,.1f}", "rank_median": "{:,.1f}", "train_p90": "{:,.1f}",
                   "rank_p90": "{:,.1f}", "avail_train": "{:.3f}", "avail_rank": "{:.3f}",
                   "median_ratio": "{:.3f}"})

In [ ]:
# Unseen categories: small, but non-zero - so an encoder needs a defined fallback, not a NaN.
rows = []
for c in ["stand_full", "stand_area", "RUNWAY_mvt", "AIRCRAFT_TYPE_mvt",
          "AIRCRAFT_OPERATOR_flt", "dest_country"]:
    seen = set(pd.read_parquet(TRAIN_F, columns=[c])[c].dropna().unique())
    rv = pd.read_parquet(RANK_F, columns=[c])[c].dropna()
    unseen = ~rv.isin(seen)
    rows.append({"column": c, "unseen_categories": rv[unseen].nunique(),
                 "rows_affected": int(unseen.sum()), "share": unseen.mean()})
pd.DataFrame(rows).style.format({"share": "{:.3%}"}).hide(axis="index")

### 13.5 Drift into 2026 is per-airport, not global

The aggregate `proxy_taxi_aobt3` median jumps from 958 s in training to 1,017 s in the ranking set, which looks
like a global slowdown. It is mostly **airport mix** — the ranking set weights airports differently from the
full training year, and July 2026 contains only EDDF, EGLL and EHAM. Comparing the *same month* year over year
separates the two.

In [ ]:
cols = ["ADEP_mvt", "MVT_TIME_UTC_mvt", "proxy_taxi_aobt3", "cong_dep_60", "queue_at_pushback"]
tt = pd.read_parquet(TRAIN_F, columns=cols + ["TAXITIME_SEC_mvt", "AIRCRAFT_TYPE_mvt"])
tt = tt[trainable_mask(tt)]
rr = pd.read_parquet(RANK_F, columns=cols)

for m, label in [(1, "JANUARY"), (7, "JULY")]:
    lhs = tt[tt["MVT_TIME_UTC_mvt"].dt.month == m].groupby("ADEP_mvt")["proxy_taxi_aobt3"].median()
    rhs = rr[rr["MVT_TIME_UTC_mvt"].dt.month == m].groupby("ADEP_mvt")["proxy_taxi_aobt3"].median()
    cmp = pd.DataFrame({f"{label[:3]}_2025": lhs, f"{label[:3]}_2026": rhs}).dropna()
    cmp["delta_s"] = cmp.iloc[:, 1] - cmp.iloc[:, 0]
    print(f"=== {label}: median proxy_taxi_aobt3, like-for-like ===")
    print(cmp.sort_values("delta_s", ascending=False).to_string(float_format=lambda x: f"{x:,.0f}"), "\n")

jan_t, jan_r = tt[tt["MVT_TIME_UTC_mvt"].dt.month == 1], rr[rr["MVT_TIME_UTC_mvt"].dt.month == 1]
print("Is it traffic volume?  (identical volume => the drift is operational, not demand)")
for c in ["cong_dep_60", "queue_at_pushback"]:
    print(f"  {c:<20} Jan-2025 {jan_t[c].median():>6.1f}   Jan-2026 {jan_r[c].median():>6.1f}"
          f"   ratio {jan_r[c].median() / jan_t[c].median():.3f}")

bias = jan_t["proxy_taxi_aobt3"].median() - jan_t["TAXITIME_SEC_mvt"].median()
implied = jan_r["proxy_taxi_aobt3"].median() - bias
print(f"\nJan-2025 target median {jan_t['TAXITIME_SEC_mvt'].median():,.0f}s"
      f"  ->  implied Jan-2026 median {implied:,.0f}s  ({implied / jan_t['TAXITIME_SEC_mvt'].median() - 1:+.1%})")

Seven of ten airports are flat within ±3 s; the real movement is **EDDF (+47 s) and EHAM (+50 s)** in January,
and **EHAM (+64 s) against EGLL (−54 s)** in July. Traffic volume is identical year over year
(`cong_dep_60` and `queue_at_pushback` both at ratio 1.000), so this is an operational change at two airports,
not a demand change.

Two consequences for the validation design in §5 of `SKILLS.md`:

1. **Never judge a validation split on its aggregate distribution** — airport mix dominates it.
2. A model keyed on `proxy_taxi_aobt3` absorbs most of this automatically, because a flight whose proxy is
   longer simply receives a longer prediction. The residual risk — the *relationship* between proxy and target
   shifting at EDDF/EHAM — is not observable from any 2025 holdout, so the final predictions deserve a
   per-airport bias audit.

### 13.6 Smoke test: does the feature set beat the baselines?

`python scripts/smoke_test.py` — CatBoost, 500 iterations, all 88 features, on a plain **Jan–Oct / Nov–Dec
2025** split.

> **This is not the stage-7 validation protocol.** The real evaluation is Jan + Jul 2026 with a lopsided
> composition (70 % January; July restricted to EDDF/EGLL/EHAM) that a scoring split has to mirror. Read the
> number below as "the feature set is wired up and carries signal", not as an expected leaderboard score.

| | RMSE on Nov–Dec 2025 |
|---|---|
| per-airport median | 377.9 s |
| raw `AOBT_3` proxy | 369.1 s |
| **CatBoost, 88 features** | **212.4 s** |

In [ ]:
imp = pd.read_csv("data/processed/smoke_importance.csv", index_col="feature")["importance"]
fam_imp = imp.groupby(imp.index.map(family)).sum().sort_values(ascending=False)

print("importance by family (%):")
print(fam_imp.to_frame("importance").to_string(float_format=lambda x: f"{x:6.2f}"))

ax = imp.head(20).sort_values().plot(kind="barh", figsize=(9, 7), color="#4C72B0")
ax.set_title("Smoke-test feature importance (CatBoost, top 20)")
ax.set_xlabel("importance (%)")
plt.tight_layout()
plt.show()

### 13.7 What this establishes

- **88 usable features** out of 115 built columns, every one of them computable on `ranking.parquet`. Five
  columns are withheld as leakage, including two — `dq_implausible_target` and `dq_was_negative` — that are
  computed from the target and were caught only at this stage.
- **The ground-geometry features are the real payoff.** `RUNWAY_mvt`, `stand_area`, `stand_full` and
  `stand_number` together carry ≈ 25 % of model importance, second only to the off-block proxies. That includes
  the stand-parsing fix: stand names take two incompatible forms (letter-prefixed `B41` at EDDF/EHAM/LFPG,
  purely numeric `583` at EGLL/EDDM/LEBL/LIRF where the leading digit is the terminal block), and a
  single-format regex silently nulled 44.5 % of rows.
- **The congestion window counts underdelivered**; the queue-state formulation is what carries that idea.
  They are kept because they are cheap and may serve in interactions, but they should not be expected to carry
  the model.
- **`ARVT_1_flt` is used narrowly and deliberately**, against the earlier blanket warning: `planned_enroute_sec`
  is a plan-side difference (`ARVT_1_flt − EOBT_1_flt`), verified not to be a post-hoc copy of the flown
  arrival — it differs from `ARVT_3_flt` by a median of −280 s and is exactly equal on only 0.07 % of rows.
- **Baselines to beat are lower than previously documented**: 373.0 s for a per-airport median on trainable
  rows, not the 519.5 s figure that included the pathological tail. See `docs/findings.md` finding 2.

Next: stages 6–7, which have to be designed together — the modelling choice depends on a validation split that
mirrors the ranking composition (70 % January, July restricted to three airports) rather than the plain time
split used above.